# DeliveryETA — Étape 1 : Chargement et nettoyage

Objectif : rendre le dataset `Food delivery.csv` exploitable pour l'EDA (étape 2) et la modélisation (étape 3).

Chaque décision de nettoyage est justifiée juste avant d'être appliquée.

## 1. Chargement et premier aperçu

In [2]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/dataset_delivery_time.csv")
print(df.shape)
df.head()

(45593, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  str    
 1   Delivery_person_ID           45593 non-null  str    
 2   Delivery_person_Age          45593 non-null  str    
 3   Delivery_person_Ratings      45593 non-null  str    
 4   Restaurant_latitude          45593 non-null  float64
 5   Restaurant_longitude         45593 non-null  float64
 6   Delivery_location_latitude   45593 non-null  float64
 7   Delivery_location_longitude  45593 non-null  float64
 8   Order_Date                   45593 non-null  str    
 9   Time_Orderd                  45593 non-null  str    
 10  Time_Order_picked            45593 non-null  str    
 11  Weatherconditions            45593 non-null  str    
 12  Road_traffic_density         45593 non-null  str    
 13  Vehicle_condition          

**Constat** : toutes les colonnes sont typées `object` (texte) sauf les 4 coordonnées GPS et `Vehicle_condition`.
Or plusieurs sont censées être numériques (`Delivery_person_Age`, `Delivery_person_Ratings`, `multiple_deliveries`,
`Time_taken(min)`), des dates (`Order_Date`) ou des heures (`Time_Orderd`, `Time_Order_picked`).
Cela vient du texte parasite présent dans les valeurs (espaces, préfixes).

In [4]:
for col in ["Weatherconditions", "Road_traffic_density", "Type_of_order",
            "Type_of_vehicle", "multiple_deliveries", "Festival", "City", "Time_taken(min)"]:
    print(col, "->", df[col].unique()[:6])

Weatherconditions -> <ArrowStringArray>
[     'conditions Sunny',     'conditions Stormy', 'conditions Sandstorms',
     'conditions Cloudy',        'conditions Fog',      'conditions Windy']
Length: 6, dtype: str
Road_traffic_density -> <ArrowStringArray>
['High ', 'Jam ', 'Low ', 'Medium ', 'NaN ']
Length: 5, dtype: str
Type_of_order -> <ArrowStringArray>
['Snack ', 'Drinks ', 'Buffet ', 'Meal ']
Length: 4, dtype: str
Type_of_vehicle -> <ArrowStringArray>
['motorcycle ', 'scooter ', 'electric_scooter ', 'bicycle ']
Length: 4, dtype: str
multiple_deliveries -> <ArrowStringArray>
['0', '1', '3', 'NaN ', '2']
Length: 5, dtype: str
Festival -> <ArrowStringArray>
['No ', 'Yes ', 'NaN ']
Length: 3, dtype: str
City -> <ArrowStringArray>
['Urban ', 'Metropolitian ', 'Semi-Urban ', 'NaN ']
Length: 4, dtype: str
Time_taken(min) -> <ArrowStringArray>
['(min) 24', '(min) 33', '(min) 26', '(min) 21', '(min) 30', '(min) 40']
Length: 6, dtype: str


## 2. Nettoyage du texte parasite

**Justification** :
- Espaces en fin de chaine (`"Urban "`) : creeraient de fausses modalites distinctes de `"Urban"` lors de l'encodage.
- Prefixe `"conditions "` devant la meteo : non informatif, a retirer.
- Prefixe `"(min) "` devant la cible : empeche toute conversion numerique tant qu'il est present.
- Chaine litterale `"NaN "` (dans plusieurs colonnes, dont `Weatherconditions`) : ce ne sont pas de vrais NaN
  pandas, donc `isna()` ne les detecte pas -- il faut les convertir explicitement, sinon elles seraient
  traitees comme une categorie valide (c'est l'erreur corrigee dans cette version du script : la premiere
  version ne traitait pas `Weatherconditions` ni les heures manquantes).

In [5]:
from src.preprocessing.cleaning import DataCleaner

cleaner = DataCleaner("../data/raw/dataset_delivery_time.csv")
df = cleaner.load()
df = cleaner.clean_text_noise(df)
df[["Weatherconditions", "City", "Time_taken(min)"]].head()

Chargement : 45593 lignes, 20 colonnes.
Texte parasite retiré (espaces, préfixes 'conditions', '(min)').


,Weatherconditions,City,Time_taken(min)
0,Sunny,Urban,24.0
1,Stormy,Metropolitian,33.0
2,Sandstorms,Urban,26.0
3,Sunny,Metropolitian,21.0
4,Cloudy,Metropolitian,30.0


## 3. Conversion des types

**Justification** : une fois le texte parasite retire, on peut convertir :
- `Delivery_person_Age`, `Delivery_person_Ratings`, `multiple_deliveries` -> numerique (`errors="coerce"`
  transforme les residus non convertibles en NaN plutot que de planter le pipeline).
- `Order_Date` -> date (format `jj-mm-aaaa` observe dans les donnees).
- `Time_Orderd`, `Time_Order_picked` -> heure, necessaires pour calculer une eventuelle duree de preparation
  en feature engineering (etape 2).
- Les chaines litterales `"NaN "` de `Road_traffic_density`, `multiple_deliveries`, `Festival`, `City` et
  **`Weatherconditions`** sont converties en vrai NaN pandas ici, sinon `handle_missing` ne les detecterait
  pas.

In [6]:
df = cleaner.convert_types(df)
df.dtypes

Types convertis : numériques (Age, Ratings, multiple_deliveries), dates et heures.


ID                                        str
Delivery_person_ID                        str
Delivery_person_Age                   float64
Delivery_person_Ratings               float64
Restaurant_latitude                   float64
Restaurant_longitude                  float64
Delivery_location_latitude            float64
Delivery_location_longitude           float64
Order_Date                     datetime64[us]
Time_Orderd                            object
Time_Order_picked                      object
Weatherconditions                         str
Road_traffic_density                      str
Vehicle_condition                       int64
Type_of_order                             str
Type_of_vehicle                           str
multiple_deliveries                   float64
Festival                                  str
City                                      str
Time_taken(min)                       float64
dtype: object

## 4. Valeurs manquantes

**Constat** (avant imputation) :

In [7]:
df.isna().sum()[df.isna().sum() > 0]

Delivery_person_Age        1854
Delivery_person_Ratings    1908
Time_Orderd                1731
Weatherconditions           616
Road_traffic_density        601
multiple_deliveries         993
Festival                    228
City                       1200
dtype: int64

**Justification des choix d'imputation** :

| Colonne | Strategie | Pourquoi |
|---|---|---|
| `Delivery_person_Age`, `Delivery_person_Ratings` | Mediane | Numeriques, distribution resserree mais avec quelques valeurs extremes ; la mediane est robuste aux outliers et ne deforme pas la distribution comme le ferait la moyenne. |
| `Road_traffic_density`, `City`, `Festival`, `multiple_deliveries`, `Weatherconditions` | Mode | Categorielles a faible cardinalite (3 a 7 modalites) ; la modalite la plus frequente est une estimation raisonnable et le volume de manquants (1 a 3 %) est trop faible pour justifier une suppression de lignes. |
| `Time_Orderd`, `Time_Order_picked` | Mode | Variable d'horaire a faible variance ici (creneaux de repas recurrents) ; reutilisee en feature engineering (duree = pickup - order), la laisser vide propagerait le NaN plus loin dans le pipeline. |

On ne supprime aucune ligne pour cause de valeur manquante : le volume perdu (jusqu'a ~4 % sur `Ratings`)
serait trop couteux pour un gain de fiabilite marginal.

In [9]:
df = cleaner.handle_missing(df)
df.isna().sum()

Delivery_person_Age : 0 valeurs manquantes imputées par la médiane (30.0).
Delivery_person_Ratings : 0 valeurs manquantes imputées par la médiane (4.7).
Road_traffic_density : 0 valeurs manquantes imputées par le mode ('Low').
City : 0 valeurs manquantes imputées par le mode ('Metropolitian').
Festival : 0 valeurs manquantes imputées par le mode ('No').
multiple_deliveries : 0 valeurs manquantes imputées par le mode ('1.0').
Weatherconditions : 0 valeurs manquantes imputées par le mode ('Fog').
Time_Orderd : 0 valeurs manquantes imputées par le mode (21:55:00).
Time_Order_picked : 0 valeurs manquantes imputées par le mode (21:30:00).


ID                             0
Delivery_person_ID             0
Delivery_person_Age            0
Delivery_person_Ratings        0
Restaurant_latitude            0
Restaurant_longitude           0
Delivery_location_latitude     0
Delivery_location_longitude    0
Order_Date                     0
Time_Orderd                    0
Time_Order_picked              0
Weatherconditions              0
Road_traffic_density           0
Vehicle_condition              0
Type_of_order                  0
Type_of_vehicle                0
multiple_deliveries            0
Festival                       0
City                           0
Time_taken(min)                0
dtype: int64

## 5. Valeurs aberrantes

**Constat sur les coordonnees GPS** :

In [10]:
df[["Restaurant_latitude", "Restaurant_longitude",
    "Delivery_location_latitude", "Delivery_location_longitude"]].describe()

,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude
count,45593.000000,45593.000000,45593.000000,45593.000000
mean,17.017729,70.231332,17.465186,70.845702
std,8.185109,22.883647,7.335122,21.118812
min,-30.905562,-88.366217,0.010000,0.010000
25%,12.933284,73.170000,12.988453,73.280000
50%,18.546947,75.898497,18.633934,76.002574
75%,22.728163,78.044095,22.785049,78.107044
max,30.914057,88.433452,31.054057,88.563452


**Justification** : certaines latitudes/longitudes sont negatives alors que toutes les villes du jeu de
donnees sont en Inde (hemisphere nord/est, valeurs attendues positives). Il s'agit d'une erreur de signe a la
saisie et non de points aberrants reels : on applique une valeur absolue plutot que de supprimer ces lignes
(qui representent une part significative du dataset).

**Coordonnees a (0, 0)** : ce point tombe dans le golfe de Guinee (Atlantique), impossible pour une livraison
en Inde. Contrairement au cas precedent, ce n'est pas une erreur de signe reparable : c'est un placeholder de
valeur manquante (~8 % des lignes). Ici on supprime les lignes plutot que de les "corriger", car on ne peut
pas deviner la vraie position.

**Constat sur la cible `Time_taken(min)`** : on utilise la methode IQR (1.5x) pour reperer les temps de
livraison irrealistes (quelques minutes ou plusieurs heures), probablement des erreurs de saisie plutot que
de vraies livraisons -- ceux-la sont supprimes car ils fausseraient l'apprentissage du modele de regression.

In [11]:
df = cleaner.handle_outliers(df)
df["Time_taken(min)"].describe()

Restaurant_latitude : 431 valeurs négatives corrigées (valeur absolue).
Restaurant_longitude : 162 valeurs négatives corrigées (valeur absolue).
Delivery_location_latitude : 0 valeurs négatives corrigées (valeur absolue).
Delivery_location_longitude : 0 valeurs négatives corrigées (valeur absolue).
Coordonnées (0, 0) : 3640 lignes supprimées (point invalide, hors Inde).
Time_taken(min) : 247 valeurs aberrantes retirées (bornes IQR [-0.5, 51.5]).


count    41706.000000
mean        26.153191
std          9.179161
min         10.000000
25%         19.000000
50%         25.000000
75%         32.000000
max         51.000000
Name: Time_taken(min), dtype: float64

## 6. Doublons

**Justification** : on verifie les doublons sur toutes les colonnes sauf `ID` (identifiant unique par nature,
qui ne doit pas empecher de detecter une commande dupliquee par erreur).

In [12]:
df = cleaner.drop_duplicates(df)
df.shape

Doublons (hors ID) : 0 lignes supprimées.


(41706, 20)

## 7. Sauvegarde du dataset nettoye

Le dataset nettoye est sauvegarde separement du brut, pour garder une trace du fichier source et permettre de
reproduire le nettoyage a tout moment.

In [13]:
cleaner.save(df, "../data/processed/dataset_delivery_time_clean.csv")

Sauvegardé : ../data/processed/dataset_delivery_time_clean.csv


## 8. Bilan du nettoyage

In [14]:
for line in cleaner.report:
    print("-", line)

- Chargement : 45593 lignes, 20 colonnes.
- Texte parasite retiré (espaces, préfixes 'conditions', '(min)').
- Types convertis : numériques (Age, Ratings, multiple_deliveries), dates et heures.
- Delivery_person_Age : 1854 valeurs manquantes imputées par la médiane (30.0).
- Delivery_person_Ratings : 1908 valeurs manquantes imputées par la médiane (4.7).
- Road_traffic_density : 601 valeurs manquantes imputées par le mode ('Low').
- City : 1200 valeurs manquantes imputées par le mode ('Metropolitian').
- Festival : 228 valeurs manquantes imputées par le mode ('No').
- multiple_deliveries : 993 valeurs manquantes imputées par le mode ('1.0').
- Weatherconditions : 616 valeurs manquantes imputées par le mode ('Fog').
- Time_Orderd : 1731 valeurs manquantes imputées par le mode (21:55:00).
- Time_Order_picked : 0 valeurs manquantes imputées par le mode (21:30:00).
- Delivery_person_Age : 0 valeurs manquantes imputées par la médiane (30.0).
- Delivery_person_Ratings : 0 valeurs manquantes 

## 9. Verification finale -- plus aucune valeur manquante

In [15]:
assert df.isna().sum().sum() == 0, "Il reste des valeurs manquantes !"
for c in df.select_dtypes(include="object").columns:
    n = df[c].astype(str).str.strip().eq("NaN").sum()
    assert n == 0, f"Il reste des 'NaN' texte dans {c}"
print("OK : aucune valeur manquante, aucun 'NaN' texte residuel.")

OK : aucune valeur manquante, aucun 'NaN' texte residuel.


C:\Users\safiy\AppData\Local\Temp\ipykernel_26548\422526645.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include="object").columns:
